In [10]:
from langgraph.graph import StateGraph, START, END

from langchain_groq import ChatGroq
model = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=""
)

from typing import TypedDict
from dotenv import load_dotenv

load_dotenv()



class BlogState(TypedDict):
    title: str
    outline: str
    content: str


def create_outline(state: BlogState) -> BlogState:
    # fetch title
    title = state['title']

    # call llm to generate outline
    prompt = f'Generate a detailed outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content

    # update state
    state['outline'] = outline

    return state


def create_blog(state: BlogState) -> BlogState:
    title = state['title']
    outline = state['outline']

    prompt = f'Write a detailed blog on the title - {title} using the following outline \n {outline}'

    content = model.invoke(prompt).content

    state['content'] = content

    return state


graph = StateGraph(BlogState)

# nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

# edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()

initial_state = {'title': 'Rise of AI in India'}

final_state = workflow.invoke(initial_state)

print(final_state)
print(final_state['outline'])
print(final_state['content'])

{'title': 'Rise of AI in India', 'outline': 'Here\'s a detailed outline for a blog on the topic "Rise of AI in India":\n\n**I. Introduction**\n\n* Brief overview of Artificial Intelligence (AI) and its growing importance globally\n* Context: India\'s emerging role in the global AI landscape\n* Thesis statement: India is experiencing a significant rise in AI adoption, driven by government initiatives, technological advancements, and a growing talent pool.\n\n**II. Government Initiatives and Policies**\n\n* Overview of key government initiatives:\n\t+ National AI Strategy (2018)\n\t+ New Education Policy (2020) - emphasis on AI and emerging technologies\n\t+ Digital India initiative\n\t+ Make in India program\n* Discussion of policy frameworks and regulations supporting AI growth in India\n* Examples of government-funded AI projects and research institutions\n\n**III. AI Adoption in Indian Industries**\n\n* Overview of AI adoption in various sectors:\n\t+ Healthcare: AI-powered diagnosis